# Cyberbullying Detection — IEEE DataPort
## Seed Ensembling: Combining Multiple BERT Models for Stable Predictions
**The novelty being tested here:** earlier multi-seed experiments showed that a single BERT model's performance (especially on the minority Cyberstalking class) varies substantially depending on random seed (std as high as ±6-8 points). This notebook tests whether **combining multiple independently-trained models via majority voting** produces predictions that are more stable and reliable than any single model — a direct, principled response to the seed-sensitivity problem discovered earlier in this project.

**Design note:** unlike the earlier multi-seed robustness notebooks, this notebook uses **one fixed train/val/test split** (so all models predict on the exact same test examples, which is required for majority voting to be valid). Only each model's weight initialization and training batch order differ across the 5 seeds — the data itself does not change.

## 1. Setup

In [ ]:
!pip install -q emoji contractions imbalanced-learn
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import time
import random
from collections import Counter

import emoji
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import RandomOverSampler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

sns.set_style("whitegrid")
plt.rc("figure", autolayout=True)
plt.rc("axes", labelweight="bold", labelsize="large", titleweight="bold", titlepad=10)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DATA_SPLIT_SEED = 2042  # fixed for everyone -- this is what makes the test set identical across models

## 2. Load + clean IEEE DataPort (done ONCE)

In [ ]:
base_path = '/kaggle/input/datasets/sudhanshuchauhan29/cyber-bullying-dataset/'

df = pd.read_csv(base_path + 'IEEE Data Port.csv', encoding='latin1')
df = df.rename(columns={'Tweet': 'text', 'Class': 'sentiment'})
df = df[~df.duplicated()]
print(df.shape)

In [ ]:
def strip_emoji(text):
    if not isinstance(text, str):
        text = str(text)
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def strip_all_entities(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'\r|\n', ' ', text.lower())
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]', '', text)
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

def clean_hashtags(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    new_tweet = re.sub(r'(\s+#[\w-]+)+\s*$', '', tweet).strip()
    new_tweet = re.sub(r'#([\w-]+)', r'\1', new_tweet).strip()
    return new_tweet

def filter_chars(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join('' if ('$' in word) or ('&' in word) else word for word in text.split())

def remove_mult_spaces(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r"\s\s+", " ", text)

def expand_contractions(text):
    if not isinstance(text, str):
        text = str(text)
    return contractions.fix(text)

def remove_numbers(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'\d+', '', text)

def lemmatize(text):
    if not isinstance(text, str):
        text = str(text)
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w) for w in words)

def remove_short_words(text, min_len=2):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(w for w in text.split() if len(w) >= min_len)

def replace_elongated_words(text):
    if not isinstance(text, str):
        text = str(text)
    regex_pattern = r'\b(\w+)((\w)\3{2,})(\w*)\b'
    return re.sub(regex_pattern, r'\1\3\4', text)

def remove_repeated_punctuation(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[\?\.\!]+(?=[\?\.\!])', '', text)

def remove_extra_whitespace(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(text.split())

def remove_url_shorteners(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(
        r'(?:http[s]?://)?(?:www\.)?(?:bit\.ly|goo\.gl|t\.co|tinyurl\.com|tr\.im|is\.gd|'
        r'cli\.gs|u\.nu|url\.ie|tiny\.cc|alturl\.com|ow\.ly|bit\.do|adoro\.to)\S+', '', text)

def remove_spaces_tweets(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    return tweet.strip()

def remove_short_tweets(tweet, min_words=3):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    words = tweet.split()
    return tweet if len(words) >= min_words else ""

def clean_tweet(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    tweet = strip_emoji(tweet)
    tweet = expand_contractions(tweet)
    tweet = strip_all_entities(tweet)
    tweet = clean_hashtags(tweet)
    tweet = filter_chars(tweet)
    tweet = remove_mult_spaces(tweet)
    tweet = remove_numbers(tweet)
    tweet = lemmatize(tweet)
    tweet = remove_short_words(tweet)
    tweet = replace_elongated_words(tweet)
    tweet = remove_repeated_punctuation(tweet)
    tweet = remove_extra_whitespace(tweet)
    tweet = remove_url_shorteners(tweet)
    tweet = remove_spaces_tweets(tweet)
    tweet = ' '.join(tweet.split())
    return tweet

df['text_clean'] = [clean_tweet(t) for t in df['text']]
df.drop_duplicates('text_clean', inplace=True)
df = df[df['text_clean'].str.len() > 0]

sentiment = ["Cyberstalking", "Revenge Porn", "Doxing", "Sexual Harassment", "Slut Shaming"]

df['text_len'] = [len(t.split()) for t in df['text_clean']]
df = df[df['text_len'] < df['text_len'].quantile(0.995)]
df['sentiment'] = df['sentiment'].replace(
    {'Cyberstalking': 0, 'Revenge Porn': 1, 'Doxing': 2, 'Sexual Harassment': 3, 'Slut Shaming': 4}
)
print(f"Final dataset size: {len(df)}")

X_all = df['text_clean'].values
y_all = df['sentiment'].values

## 3. ONE fixed train/val/test split (used by ALL 5 models)
This is the key structural difference from earlier notebooks: the split happens once, using `DATA_SPLIT_SEED`, and is reused for every model. Only each model's own training seed (weight init + batch order) varies.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=DATA_SPLIT_SEED)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=DATA_SPLIT_SEED)

ros = RandomOverSampler(random_state=DATA_SPLIT_SEED)
X_train_res, y_train_res = ros.fit_resample(
    np.array(X_train).reshape(-1, 1), np.array(y_train).reshape(-1, 1))
X_train = X_train_res.flatten()
y_train = y_train_res.flatten()

print(f"Train: {len(X_train)} | Val: {len(X_valid)} | Test: {len(X_test)}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
MAX_LEN = 128

def bert_tokenizer(data):
    input_ids, attention_masks = [], []
    for sent in data:
        encoded_sent = tokenizer(
            sent, add_special_tokens=True, max_length=MAX_LEN,
            padding='max_length', truncation=True, return_attention_mask=True
        )
        input_ids.append(encoded_sent['input_ids'])
        attention_masks.append(encoded_sent['attention_mask'])
    return torch.tensor(input_ids), torch.tensor(attention_masks)

train_inputs, train_masks = bert_tokenizer(X_train)
val_inputs, val_masks = bert_tokenizer(X_valid)
test_inputs, test_masks = bert_tokenizer(X_test)

train_labels = torch.tensor(y_train, dtype=torch.long)
val_labels = torch.tensor(y_valid, dtype=torch.long)
test_labels = torch.tensor(y_test, dtype=torch.long)

batch_size = 32
train_data = TensorDataset(train_inputs, train_masks, train_labels)
val_data = TensorDataset(val_inputs, val_masks, val_labels)
test_data = TensorDataset(test_inputs, test_masks, test_labels)

val_dataloader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=batch_size)
test_dataloader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=batch_size)

## 4. Model definition — Baseline BERT (CLS-only), the strongest performer so far

In [ ]:
N_OUTPUT = 5

class Bert_Classifier_CLS(nn.Module):
    def __init__(self, freeze_bert=False):
        super().__init__()
        n_hidden = 50
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden), nn.ReLU(), nn.Linear(n_hidden, N_OUTPUT)
        )
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = outputs[0][:, 0, :]
        return self.classifier(cls_vec)

## 5. Training / evaluation functions

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def initialize_model(model_class, train_dataloader, epochs=10):
    model = model_class(freeze_bert=False)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    return model, optimizer, scheduler


def bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader, epochs=10, patience=3, name=""):
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch_i in range(epochs):
        t0_epoch = time.time()
        total_loss = 0
        model.train()
        for step, batch in enumerate(train_dataloader):
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            model.zero_grad()
            logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        avg_train_loss = total_loss / len(train_dataloader)

        model.eval()
        val_accuracy, val_loss = [], []
        for batch in val_dataloader:
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            with torch.no_grad():
                logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            val_loss.append(loss.item())
            preds = torch.argmax(logits, dim=1).flatten()
            val_accuracy.append((preds == b_labels).cpu().numpy().mean() * 100)
        val_loss = np.mean(val_loss)
        val_accuracy = np.mean(val_accuracy)
        elapsed = time.time() - t0_epoch
        print(f"  [{name}] Epoch {epoch_i+1:>2} | train loss {avg_train_loss:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_accuracy:.2f}% | {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch_i+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_on_test(model, test_dataloader):
    model.eval()
    preds_list = []
    for batch in test_dataloader:
        b_input_ids, b_attn_mask, _ = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            logits = model(b_input_ids, b_attn_mask)
        preds_list.extend(torch.argmax(logits, dim=1).cpu().numpy())
    return np.array(preds_list)

## 6. Train 5 models (SAME data, different training seeds) and collect predictions

In [ ]:
MODEL_SEEDS = [2042, 7, 123, 55, 999]
EPOCHS = 10

all_model_preds = []   # list of prediction arrays, one per model
all_model_accs = []

for seed in MODEL_SEEDS:
    print(f"\n{'='*60}\nTraining model with seed={seed}\n{'='*60}")
    set_all_seeds(seed)

    # Rebuild the train dataloader fresh each time so shuffling order also
    # differs per seed (this is part of what makes each model a genuinely
    # different, independent draw for ensembling purposes)
    train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=batch_size)

    model, optimizer, scheduler = initialize_model(Bert_Classifier_CLS, train_dataloader, epochs=EPOCHS)
    model = bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader,
                        epochs=EPOCHS, patience=3, name=f"seed={seed}")

    preds = predict_on_test(model, test_dataloader)
    acc = accuracy_score(y_test, preds)
    print(f"  Model seed={seed} -> test accuracy = {acc:.4f}")

    all_model_preds.append(preds)
    all_model_accs.append(acc)

    del model
    torch.cuda.empty_cache()

all_model_preds = np.array(all_model_preds)  # shape: (5, n_test_examples)
print("\nAll 5 models trained. Individual accuracies:", [f"{a:.4f}" for a in all_model_accs])

## 7. Majority-vote ensemble
For each test example, take the most common prediction across the 5 models. Ties (rare, with an odd number of models this only happens with >2 classes tied) are broken by taking the first mode found.

In [ ]:
def majority_vote(preds_matrix):
    n_examples = preds_matrix.shape[1]
    ensemble_preds = np.zeros(n_examples, dtype=int)
    for i in range(n_examples):
        votes = preds_matrix[:, i]
        ensemble_preds[i] = Counter(votes).most_common(1)[0][0]
    return ensemble_preds

ensemble_preds = majority_vote(all_model_preds)
ensemble_acc = accuracy_score(y_test, ensemble_preds)

print(f"Individual model accuracies: {[f'{a:.4f}' for a in all_model_accs]}")
print(f"Mean individual accuracy:    {np.mean(all_model_accs):.4f}")
print(f"Std individual accuracy:     {np.std(all_model_accs):.4f}")
print(f"Best single model accuracy:  {np.max(all_model_accs):.4f}")
print(f"ENSEMBLE (majority vote):    {ensemble_acc:.4f}")

## 8. Full classification report — Ensemble vs. best single model

In [ ]:
best_model_idx = int(np.argmax(all_model_accs))
best_single_preds = all_model_preds[best_model_idx]

print(f"Best single model (seed={MODEL_SEEDS[best_model_idx]}):\n")
print(classification_report(y_test, best_single_preds, target_names=sentiment, digits=3))

print(f"\n{'='*60}\nEnsemble (majority vote of all 5 models):\n")
print(classification_report(y_test, ensemble_preds, target_names=sentiment, digits=3))